# Microsoft Graph API Basics

**Goal:** Authenticate with MSAL, call the Microsoft Graph API to read your user profile, and understand how Graph connects to Power Automate — in under 2 minutes.

**Estimated time:** 90 seconds (excluding initial package install)

---

## What Is the Microsoft Graph API?

Microsoft Graph (`https://graph.microsoft.com/v1.0`) is the single REST gateway to all Microsoft 365 data: users, mail, calendar, files (OneDrive/SharePoint), Teams, and more. Every connector in Power Automate that touches Microsoft 365 data — Outlook, SharePoint, Teams, OneDrive — is calling Graph behind the scenes.

Understanding Graph gives you two superpowers:

1. **Build custom connectors** that expose Graph data to flows without Premium licensing
2. **Call Power Automate's management API** to list, trigger, and monitor flows programmatically

```
Your Python code
      │
      │  Bearer token (OAuth 2.0)
      ▼
Microsoft Graph API  ──────────────────────────────────────────┐
  /me                   (your profile)                         │
  /me/messages          (your inbox)                           │
  /sites/{id}/lists     (SharePoint lists)                     ├── All the same data
  /groups               (Teams / M365 groups)                  │   Power Automate
  /drives/{id}/items    (OneDrive / SharePoint files)          │   connectors use
                                                               │
Power Automate Management API  ────────────────────────────────┘
  api.flow.microsoft.com
  /providers/Microsoft.ProcessSimple/environments/{env}/flows
```

---

## Install Required Packages

The Microsoft Authentication Library (`msal`) handles the OAuth 2.0 token acquisition. Run this cell once per environment.

In [ ]:
import sys; sys.path.insert(0, '../../..')
from resources.notebook_style import apply_course_theme, learning_objectives, section_divider, callout, key_takeaways
from resources.graphics.plot_theme import apply_plot_theme

In [ ]:
apply_course_theme()
apply_plot_theme()

In [ ]:
# Install the Microsoft Authentication Library.
# requests is included in most Python environments; listed here for completeness.
%pip install msal requests --quiet

## Register an Azure AD Application

Before authenticating, you need an **App Registration** in Azure Active Directory (now called Microsoft Entra ID). This is a one-time setup that takes about 3 minutes.

**Steps:**
1. Go to https://portal.azure.com → **Microsoft Entra ID** → **App registrations** → **New registration**
2. Name: `Power Automate Quick Start`, Supported account types: **Single tenant**, Redirect URI: leave blank. Click **Register**.
3. Copy the **Application (client) ID** and **Directory (tenant) ID** from the Overview page.
4. Go to **Authentication** → Enable **"Allow public client flows"** → Save.
5. Go to **API permissions** → **Add a permission** → **Microsoft Graph** → **Delegated** → add `User.Read`. Click **Grant admin consent**.

Paste the IDs below.

In [ ]:
# Replace these placeholders with your actual IDs from the Azure portal.
TENANT_ID = "YOUR_TENANT_ID_HERE"          # Directory (tenant) ID
CLIENT_ID = "YOUR_CLIENT_ID_HERE"          # Application (client) ID

# Graph API endpoint
GRAPH_API_BASE = "https://graph.microsoft.com/v1.0"

# OAuth 2.0 scopes this notebook needs.
# User.Read lets us read the signed-in user's basic profile.
SCOPES = ["User.Read"]

# Validate placeholders
ready = "YOUR_TENANT_ID_HERE" not in TENANT_ID and "YOUR_CLIENT_ID_HERE" not in CLIENT_ID
if ready:
    print(f"Tenant ID : {TENANT_ID}")
    print(f"Client ID : {CLIENT_ID}")
    print(f"Scopes    : {SCOPES}")
    print(f"Graph base: {GRAPH_API_BASE}")
else:
    print("ACTION REQUIRED: Replace TENANT_ID and CLIENT_ID with your Azure app registration values.")
    print("See the instructions in the cell above.")

## Authenticate with MSAL — Device Code Flow

MSAL supports several OAuth 2.0 grant types. For interactive notebooks, the **device code flow** is the best choice: it prints a URL and a code, you open the URL in any browser and enter the code, and MSAL receives a token automatically. No redirect URI or browser automation required.

The token is cached in memory. Subsequent calls in this session reuse it without re-prompting.

In [ ]:
import msal
import requests
import json

def get_access_token(tenant_id: str, client_id: str, scopes: list[str]) -> str | None:
    """
    Acquire an OAuth 2.0 access token using the MSAL device code flow.

    The device code flow works in any environment (no browser redirect needed):
    1. MSAL returns a user_code and a verification_uri
    2. The user opens the URI and enters the code in a browser (any device)
    3. MSAL polls Azure AD and returns the token when authentication completes

    Parameters
    ----------
    tenant_id  : Azure AD tenant ID (from app registration overview)
    client_id  : Application (client) ID (from app registration overview)
    scopes     : List of MS Graph permission scopes, e.g. ["User.Read"]

    Returns
    -------
    Access token string, or None if authentication fails
    """
    # PublicClientApplication is used for flows that authenticate a real user
    # (as opposed to ConfidentialClientApplication for app-only/daemon auth).
    app = msal.PublicClientApplication(
        client_id=client_id,
        authority=f"https://login.microsoftonline.com/{tenant_id}"
    )

    # Check the in-memory token cache first — avoids re-prompting within the session.
    accounts = app.get_accounts()
    if accounts:
        result = app.acquire_token_silent(scopes, account=accounts[0])
        if result and "access_token" in result:
            print("Access token retrieved from cache.")
            return result["access_token"]

    # No cached token — initiate the device code flow.
    flow = app.initiate_device_flow(scopes=scopes)

    if "user_code" not in flow:
        print(f"Failed to initiate device flow: {flow.get('error_description', 'unknown error')}")
        return None

    # Print the instructions exactly as MSAL formats them.
    # The message field says: "To sign in, use a web browser to open ... and enter code ..."
    print(flow["message"])
    print()

    # Block and poll until the user completes sign-in or the code expires (15 min).
    result = app.acquire_token_by_device_flow(flow)

    if "access_token" in result:
        print("Authentication successful.")
        return result["access_token"]
    else:
        print(f"Authentication failed: {result.get('error_description', result.get('error', 'unknown'))}")
        return None


# Run authentication — will print a device code prompt if credentials are not cached.
if ready:
    access_token = get_access_token(TENANT_ID, CLIENT_ID, SCOPES)
else:
    access_token = None
    print("Skipping authentication — set TENANT_ID and CLIENT_ID first.")

## Call the Graph API — Get Your User Profile

With the access token, every Graph API call follows the same pattern: set an `Authorization: Bearer <token>` header and send a GET request to the endpoint. The `/me` endpoint returns the profile of the authenticated user.

In [ ]:
def call_graph_api(token: str, endpoint: str) -> dict | None:
    """
    Make an authenticated GET request to the Microsoft Graph API.

    Parameters
    ----------
    token    : OAuth 2.0 Bearer access token from MSAL
    endpoint : Graph API path, e.g. "/me" or "/me/messages?$top=5"

    Returns
    -------
    Parsed JSON response as a dict, or None on error
    """
    url = f"{GRAPH_API_BASE}{endpoint}"
    headers = {
        "Authorization": f"Bearer {token}",
        "Content-Type": "application/json"
    }

    response = requests.get(url, headers=headers, timeout=15)

    if response.status_code == 200:
        return response.json()
    else:
        print(f"Graph API error {response.status_code}: {response.text[:300]}")
        return None


if access_token:
    # /me returns the signed-in user's profile.
    # Requires the User.Read scope granted in the app registration.
    profile = call_graph_api(access_token, "/me")

    if profile:
        # Extract the fields most useful for Power Automate context
        print("Signed-in user profile:")
        print(f"  Display name : {profile.get('displayName')}")
        print(f"  Email        : {profile.get('mail') or profile.get('userPrincipalName')}")
        print(f"  Job title    : {profile.get('jobTitle', '(not set)')}")
        print(f"  User ID      : {profile.get('id')}")
        print(f"  Tenant domain: {profile.get('userPrincipalName', '').split('@')[-1]}")
        print()
        print("Full response:")
        print(json.dumps(profile, indent=2))
else:
    print("No access token available.")
    print("Authenticate in the cell above, then re-run this cell.")
    print()
    print("Example /me response shape:")
    example = {
        "displayName": "Alex Johnson",
        "mail": "alex@contoso.com",
        "userPrincipalName": "alex@contoso.onmicrosoft.com",
        "id": "87d349ed-44d7-43e1-9a83-5f2406dee5bd",
        "jobTitle": "Developer",
        "officeLocation": "London"
    }
    print(json.dumps(example, indent=2))

## How Graph Connects to Power Automate

The Power Automate management API (`api.flow.microsoft.com`) uses the same OAuth 2.0 token system as Graph. Once you have a token with the right scopes, you can use it to:

- **List all flows** in an environment
- **Get flow run history** (audit trail)
- **Enable or disable flows** programmatically
- **Export flows** as packages for deployment

The additional scope required for Power Automate management is `https://service.flow.microsoft.com/.default`. Add it to the `SCOPES` list and re-authenticate to gain access.

In [ ]:
# Demonstrate how the same token pattern reaches the Power Automate management API.
# To run this cell, re-authenticate with the expanded scope list below.

PA_MANAGEMENT_SCOPES = [
    "User.Read",                                       # Graph: read user profile
    "https://service.flow.microsoft.com/.default"      # Power Automate management
]

# The environment ID identifies your Power Platform environment.
# Find it at: https://admin.powerplatform.microsoft.com → Environments → select your env
# The default environment ID format: /providers/Microsoft.ProcessSimple/environments/Default-{TENANT_ID}
ENVIRONMENT_ID = f"Default-{TENANT_ID}"  # Replace with your actual environment ID if different

PA_API_BASE = "https://api.flow.microsoft.com"
PA_API_VERSION = "2016-11-01"

# Show the URL structure for listing flows in an environment
list_flows_url = (
    f"{PA_API_BASE}/providers/Microsoft.ProcessSimple"
    f"/environments/{ENVIRONMENT_ID}/flows"
    f"?api-version={PA_API_VERSION}"
)

print("Power Automate management API — list flows endpoint:")
print(f"  {list_flows_url}")
print()
print("To call this endpoint, authenticate with these scopes:")
for scope in PA_MANAGEMENT_SCOPES:
    print(f"  - {scope}")
print()
print("The call pattern is identical to Graph:")
print("  headers = {'Authorization': f'Bearer {access_token}'}")
print("  response = requests.get(list_flows_url, headers=headers)")

# Attempt the call if a token is available.
# This will succeed only if the token was acquired with the PA management scope.
if access_token:
    headers = {
        "Authorization": f"Bearer {access_token}",
        "Content-Type": "application/json"
    }
    response = requests.get(list_flows_url, headers=headers, timeout=15)
    print()
    print(f"List flows response: {response.status_code} {response.reason}")
    if response.status_code == 200:
        flows = response.json().get("value", [])
        print(f"Flows found in environment: {len(flows)}")
        for flow in flows[:5]:  # Show first 5 only
            print(f"  - {flow.get('properties', {}).get('displayName', '(unnamed)')}")
    elif response.status_code == 403:
        print("403 Forbidden — re-authenticate with PA_MANAGEMENT_SCOPES to list flows.")

## Scope Reference

Use this table when building integrations that combine Graph and Power Automate management.

| What You Want to Do | Scope to Add |
|---------------------|--------------|
| Read signed-in user profile | `User.Read` |
| Read user's email | `Mail.Read` |
| Send email as user | `Mail.Send` |
| Read SharePoint sites and lists | `Sites.Read.All` |
| Read OneDrive files | `Files.Read` |
| List/manage Power Automate flows | `https://service.flow.microsoft.com/.default` |
| Call Power Automate management API | `https://service.flow.microsoft.com/.default` |

All scopes must be added to the **API permissions** tab of your Azure app registration and granted admin consent before they can be requested at runtime.

In [ ]:
section_divider("Next Steps")

## Next Steps

You can now authenticate to Microsoft Graph with MSAL and read data from both Graph and the Power Automate management API.

| Notebook / Module | What You Will Do |
|-------------------|------------------|
| **[02_trigger_a_flow.ipynb](02_trigger_a_flow.ipynb)** | Trigger a flow via HTTP POST — no token needed, SAS URL handles auth |
| **[Module 00: Platform Orientation](../modules/module_00_platform_orientation/)** | Full environment setup and Power Platform overview |
| **[Module 02: Triggers and Connectors](../modules/module_02_triggers_connectors/)** | Deep dive into connectors — which ones call Graph, which need custom auth |